<a href="https://colab.research.google.com/github/iguchi-lab/Verification-Platform/blob/feat/ipywidgets-tab-ui/Verification_Platform_default.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 必要なモジュールのインストール

!pip3 install git+https://github.com/iguchi-lab/pyhees-jjj.git@jjj-experiment-v3.8
#!pip3 install git+https://github.com/izumi-system-development/pyhees-jjj.git@jjj-experiment-develop

#以下は、ライブラリをアップグレードする場合に実行する。
#!pip3 install pandas --upgrade
#!pip3 install numpy --upgrade
#!pip3 install scipy --upgrade
#!pip3 install setuptools --upgrade
#!pip3 install jedi

## ユーザーマニュアル
[ユーザーマニュアル 計算フロー を開く](https://iguchi-lab.github.io/pyhees-jjj/%E8%A8%88%E7%AE%97%E3%83%95%E3%83%AD%E3%83%BC_%E3%82%BF%E3%82%A4%E3%83%971%E3%83%BB2.html)

In [ ]:
#@title (1) タブUIを使って入力する場合
# 既存のColabフォーム定義から、ipywidgetsのタブUIを自動生成します。

import ast
import base64
import html
import json
import re
import traceback
from collections import OrderedDict, defaultdict

import ipywidgets as widgets
import jjjexperiment.main
from IPython.display import clear_output, display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except (ImportError, AttributeError):
    pass

_FORM_SOURCE = base64.b64decode("...").decode("utf-8")

_ASSIGNMENT_RE = re.compile(
    r"^(\s*)([A-Za-z_]\w*)\s*=\s*(.*?)\s+#@para"
    + r"m\s*(.*)$"
)

def _plain_label(value):
    value = re.sub(r"<[^>]+>", "", value)
    value = value.replace("#@markdown", "")
    value = value.replace("**", "").replace("#", "")
    return html.unescape(value).strip(" -―")

def _safe_literal(expression):
    try:
        return ast.literal_eval(expression)
    except (ValueError, SyntaxError):
        return expression.strip("'\"")

def _extract_options(meta):
    start = meta.find("[")
    end = meta.rfind("]")
    if start < 0 or end <= start:
        return None
    try:
        return list(ast.literal_eval(meta[start:end + 1]))
    except (ValueError, SyntaxError):
        return None

def _extract_fields(source):
    fields = []
    occurrences = defaultdict(int)
    section = "① 計算条件名"
    group = "基本項目"
    label = None

    for line in source.splitlines():
        if "#@markdown" in line:
            cleaned = _plain_label(line)
            section_match = re.search(r"＜([^＞]+)＞", cleaned)
            if section_match:
                section = section_match.group(1).strip()
                group = "基本項目"
                label = None
            elif "―" in line and cleaned:
                group = cleaned
                label = None
            elif cleaned:
                label = cleaned

        match = _ASSIGNMENT_RE.match(line)
        if not match:
            continue

        _, name, expression, meta = match.groups()
        occurrence = occurrences[name]
        occurrences[name] += 1

        fields.append({
            "id": f"{name}__{occurrence}",
            "name": name,
            "occurrence": occurrence,
            "label": label or name,
            "section": section,
            "group": group,
            "default": _safe_literal(expression),
            "options": _extract_options(meta),
            "meta": meta,
        })

    return fields

def _make_control(field):
    default = field["default"]
    options = field["options"]
    common = {
        "description": "",
        "layout": widgets.Layout(width="360px"),
    }

    if options:
        if default not in options and options:
            default = options[0]
        control = widgets.Dropdown(options=options, value=default, **common)
    elif isinstance(default, bool):
        control = widgets.Checkbox(value=default, indent=False, **common)
    elif isinstance(default, int) and not isinstance(default, bool):
        control = widgets.IntText(value=default, **common)
    elif isinstance(default, float):
        control = widgets.FloatText(value=default, **common)
    else:
        control = widgets.Text(value=str(default), **common)

    control.tooltip = field["id"]
    return control

def _field_row(field, control):
    variable_text = field["name"]
    if field["occurrence"]:
        variable_text += f"（出現 {field['occurrence'] + 1}）"

    label = widgets.HTML(
        value=(
            f"<div style='line-height:1.25'>"
            f"<b>{html.escape(field['label'])}</b>"
            f"<br><code>{html.escape(variable_text)}</code></div>"
        ),
        layout=widgets.Layout(width="470px"),
    )
    return widgets.HBox(
        [label, control],
        layout=widgets.Layout(
            width="100%",
            align_items="center",
            border_bottom="1px solid #eee",
            padding="6px 2px",
        ),
    )

def _main_category(section):
    if section.startswith("⑦") or "暖房全般" in section:
        return "暖房"
    if section.startswith("⑧") or "冷房全般" in section:
        return "冷房"
    if section.startswith("⑨") or "換気" in section:
        return "換気"
    return "基本設定"

_FIELDS = _extract_fields(_FORM_SOURCE)
_CONTROLS = {field["id"]: _make_control(field) for field in _FIELDS}

if len(_FIELDS) != len(_CONTROLS):
    raise RuntimeError(
        f"入力定義数とウィジェット数が一致しません: "
        f"{len(_FIELDS)} != {len(_CONTROLS)}"
    )

def _category_panel(category):
    sections = OrderedDict()
    for field in _FIELDS:
        if _main_category(field["section"]) == category:
            sections.setdefault(field["section"], []).append(field)

    items = []
    for section_name, section_fields in sections.items():
        items.append(
            widgets.HTML(
                value=(
                    "<div style='margin:12px 0 6px;"
                    "padding:8px 10px;background:#e8f0fe;"
                    "border-left:5px solid #4285f4'>"
                    f"<b>{html.escape(section_name)}</b></div>"
                )
            )
        )

        current_group = None
        for field in section_fields:
            if field["group"] != current_group:
                current_group = field["group"]
                if current_group and current_group != "基本項目":
                    items.append(
                        widgets.HTML(
                            value=(
                                "<div style='margin:8px 0 2px;"
                                "font-weight:bold;color:#5f6368'>"
                                f"{html.escape(current_group)}</div>"
                            )
                        )
                    )
            items.append(
                _field_row(field, _CONTROLS[field["id"]])
            )

    if not items:
        items.append(widgets.HTML("<i>入力項目はありません。</i>"))

    return widgets.VBox(
        items,
        layout=widgets.Layout(
            width="100%",
            max_height="720px",
            overflow_y="auto",
            border="1px solid #dadce0",
            padding="8px",
        ),
    )

_CATEGORIES = ["基本設定", "暖房", "冷房", "換気"]
_CATEGORY_PANELS = {
    category: _category_panel(category)
    for category in _CATEGORIES
}

# ColabではTabの見出しが表示されない場合があるため、
# ToggleButtonsをタブバーとして使用します。
_CATEGORY_COUNTS = {
    category: sum(
        1 for field in _FIELDS
        if _main_category(field["section"]) == category
    )
    for category in _CATEGORIES
}
_category_selector = widgets.ToggleButtons(
    options=[
        (f"{category} ({_CATEGORY_COUNTS[category]})", category)
        for category in _CATEGORIES
    ],
    value="基本設定",
    description="",
    button_style="info",
    layout=widgets.Layout(width="100%"),
)
_category_host = widgets.VBox(
    [_CATEGORY_PANELS["基本設定"]],
    layout=widgets.Layout(width="100%"),
)

def _switch_category(change):
    if change["name"] == "value" and change["new"]:
        _category_host.children = (
            _CATEGORY_PANELS[change["new"]],
        )

_category_selector.observe(_switch_category, names="value")

def _build_input_data(values):
    transformed = []
    occurrences = defaultdict(int)

    for line in _FORM_SOURCE.splitlines():
        if "jjjexperiment.main.calc(input_data)" in line:
            continue

        match = _ASSIGNMENT_RE.match(line)
        if match:
            indent, name, expression, _ = match.groups()
            occurrence = occurrences[name]
            occurrences[name] += 1
            field_id = f"{name}__{occurrence}"
            line = f"{indent}{name} = _values.get({field_id!r}, {expression})"

        transformed.append(line)

    namespace = {"_values": values}
    exec("\n".join(transformed), namespace)
    return namespace["input_data"]

def _current_values():
    return {
        field["id"]: _CONTROLS[field["id"]].value
        for field in _FIELDS
    }

def _all_input_report(values):
    report = OrderedDict()
    for field in _FIELDS:
        section = report.setdefault(field["section"], OrderedDict())
        group = section.setdefault(field["group"], [])
        group.append({
            "id": field["id"],
            "label": field["label"],
            "variable": field["name"],
            "value": values[field["id"]],
        })
    return report

_preview_all = widgets.Textarea(
    description="",
    disabled=False,
    layout=widgets.Layout(width="100%", height="520px"),
)
_preview_effective = widgets.Textarea(
    description="",
    disabled=False,
    layout=widgets.Layout(width="100%", height="520px"),
)
_preview_selector = widgets.ToggleButtons(
    options=[
        ("全222入力", "all"),
        ("計算用 input_data", "effective"),
    ],
    value="all",
    description="",
    button_style="info",
)
_preview_host = widgets.VBox(
    [_preview_all],
    layout=widgets.Layout(width="100%"),
)
_preview_box = widgets.VBox(
    [_preview_selector, _preview_host],
    layout=widgets.Layout(width="100%", display="none"),
)

def _switch_preview(change):
    if change["name"] != "value":
        return
    selected = (
        _preview_all
        if change["new"] == "all"
        else _preview_effective
    )
    _preview_host.children = (selected,)

_preview_selector.observe(_switch_preview, names="value")

_status = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ddd",
        padding="8px",
        width="100%",
    )
)

_check_button = widgets.Button(
    description="入力内容を確認",
    icon="check",
)
_run_button = widgets.Button(
    description="計算を実行",
    button_style="primary",
    icon="play",
)

def _check_inputs(_):
    global input_data
    with _status:
        clear_output()
        try:
            values = _current_values()
            input_data = _build_input_data(values)

            _preview_all.value = json.dumps(
                _all_input_report(values),
                ensure_ascii=False,
                indent=2,
                default=str,
            )
            _preview_effective.value = json.dumps(
                input_data,
                ensure_ascii=False,
                indent=2,
                default=str,
            )
            _preview_box.layout.display = ""
            _preview_selector.value = "all"
            _preview_host.children = (_preview_all,)

            print(
                f"入力内容を作成しました: "
                f"{len(_FIELDS)}定義 / {len(_CONTROLS)}ウィジェット"
            )
            print(
                "「全222入力」と「計算用 input_data」を切り替えて"
                "内容を確認できます。"
            )
        except Exception:
            traceback.print_exc()

def _run_calculation(_):
    global input_data
    with _status:
        clear_output()
        try:
            values = _current_values()
            input_data = _build_input_data(values)
            print("計算を開始します...")
            jjjexperiment.main.calc(input_data)
            print("計算が完了しました。")
        except Exception:
            traceback.print_exc()

_check_button.on_click(_check_inputs)
_run_button.on_click(_run_calculation)

_header = widgets.HTML(
    "<h2>Verification Platform 入力</h2>"
    "<p>タブを選び、表示されたチェックボックス、選択欄、"
    "数値欄を直接編集してください。"
    "既存の全入力項目と既定値を引き継いでいます。</p>"
)
_buttons = widgets.HBox([_check_button, _run_button])

display(
    widgets.VBox([
        _header,
        _category_selector,
        _category_host,
        _buttons,
        _status,
        _preview_box,
    ])
)


In [ ]:
#@title (2) JSONファイルをアップロードして実行する場合
import json
import jjjexperiment.main
from google.colab import files
uploaded = files.upload()
with open(list(uploaded.keys())[0]) as f:
    input_data = json.load(f)
jjjexperiment.main.calc(input_data)

In [ ]:
#@title (3) /content/にアップロードされたJSONファイルをすべて実行する場合
import jjjexperiment.main
import glob
import json

filepath = glob.glob('/content/*json')
for fn in filepath:
    print('calc: ' + fn)
    with open(fn) as f:
        input_data = json.load(f)
    jjjexperiment.main.calc(input_data)

In [4]:
#@title (4) 計算結果のグラフ表示

!pip install japanize-matplotlib                                        #日本語mtaplotlibモジュール

import numpy as np
import pandas as pd
import japanize_matplotlib
import matplotlib.pyplot as plt
from jjjexperiment.constants import version_info

prefix_filename = input_data['case_name'] + version_info()

df_output2   = pd.read_csv(prefix_filename + '_output2.csv',   encoding = 'cp932', parse_dates = True, index_col = 0)
df_output5_C = pd.read_csv(prefix_filename + '_C_output5.csv', encoding = 'cp932', parse_dates = True, index_col = 0)
df_output5_H = pd.read_csv(prefix_filename + '_H_output5.csv', encoding = 'cp932', parse_dates = True, index_col = 0)

winter_df_output2   =   df_output2.loc['2023-02-05 00:00:00' : '2023-02-13 00:00:00']
winter_df_output5_H = df_output5_H.loc['2023-02-05 00:00:00' : '2023-02-13 00:00:00']
summer_df_output2   =   df_output2.loc['2023-08-06 00:00:00' : '2023-08-14 00:00:00']
summer_df_output5_C = df_output5_C.loc['2023-08-06 00:00:00' : '2023-08-14 00:00:00']

sorted_H_df_output2 = df_output2.sort_values('q_hs_H_d_t [Wh/h]', ascending = False)
sorted_H_df_output2.loc[sorted_H_df_output2['q_hs_H_d_t [Wh/h]'] <= 0.0] = np.nan

df_output2['q_hs_C_d_t [Wh/h]'] = df_output2['q_hs_CS_d_t [Wh/h]'] + df_output2['q_hs_CL_d_t [Wh/h]']
sorted_C_df_output2 = df_output2.sort_values('q_hs_C_d_t [Wh/h]', ascending = False)
sorted_C_df_output2['q_hs_C_d_t [Wh/h]'] = sorted_C_df_output2['q_hs_CS_d_t [Wh/h]'] + sorted_C_df_output2['q_hs_CL_d_t [Wh/h]']
sorted_C_df_output2.loc[sorted_C_df_output2['q_hs_C_d_t [Wh/h]'] <= 0.0] = np.nan

####Heating#############################################################################################################

plt.figure(figsize=(25, 7))

plt.subplot(2, 2, 1)
plt.plot(winter_df_output2.index,
         winter_df_output2["Theta_ex_d_t [℃]"],
         label = "外気温[℃]", color = 'lightblue')
plt.plot(winter_df_output5_H.index,
         winter_df_output5_H["X_ex_d_t"] * 1000,
         label = "外気絶対湿度[g/kg']", color = 'orange')
plt.ylabel("温度[℃]、絶対湿度[g/kg']")
plt.ylim(-10, 40)
plt.grid()
plt.legend()

plt.subplot(2, 2, 2)
plt.plot(winter_df_output2.index,
         winter_df_output2["q_hs_H_d_t [Wh/h]"] / 1000,
         label = "処理熱量[kWh/h]", color = 'red')
plt.plot(winter_df_output2.index,
         winter_df_output2["E_E_H_d_t [kWh/h]"],
         label = "AC+FAN消費電力[kWh/h]", color = 'blue')
plt.plot(winter_df_output2.index,
         winter_df_output2["E_E_fan_H_d_t [kWh/h]"],
         label = "FAN消費電力[kWh/h]", color = 'green')
plt.ylabel("処理熱量[kWh/h]、消費電力[kWh/h]")
plt.ylim(0, 10)
plt.grid()
plt.legend()

plt.subplot(2, 2, 3)
plt.plot(winter_df_output2.index,
         winter_df_output2["E_UT_H_d_t [MJ/h]"],
         label = "処理熱量[kWh/h]", color = 'red')
plt.ylabel("未処理負荷（一次エネルギー相当分）[MJ/h]")
plt.ylim(0, 10)
plt.grid()
plt.legend()

plt.subplot(2, 2, 4)
plt.plot(winter_df_output2.index,
         winter_df_output2["q_hs_H_d_t [Wh/h]"] / 1000 / winter_df_output2["E_E_H_d_t [kWh/h]"],
         label = "sCOP[-]", color = 'black')
plt.ylabel("sCOP[-]")
plt.ylim(0, 10)
plt.grid()
plt.legend()
plt.show()

########################################################################################################################

plt.figure(figsize=(25, 7))

plt.subplot(2, 4, 1)
plt.plot(range(8760), sorted_H_df_output2['q_hs_H_d_t [Wh/h]'] / 1000, color = 'orange')
plt.ylabel("処理熱量[kWh/h]")
plt.xlabel("時間[h]")
plt.ylim(0, 10)
plt.grid()

plt.subplot(2, 4, 2)
plt.plot(range(8760), sorted_H_df_output2['Theta_hs_H_out_d_t [℃]'], color = 'orange')
plt.ylabel("室内機出口温度[℃]")
plt.xlabel("時間[h]")
plt.ylim(0, 50)
plt.grid()

plt.subplot(2, 4, 3)
plt.plot(range(8760), sorted_H_df_output2['Theta_hs_H_in_d_t [℃]'], color = 'orange')
plt.ylabel("室内機入口温度[℃]")
plt.xlabel("時間[h]")
plt.ylim(0, 50)
plt.grid()

plt.subplot(2, 4, 4)
plt.plot(range(8760), sorted_H_df_output2['V_hs_supply_H_d_t [m3/h]'], color = 'orange')
plt.ylabel("供給風量[m3/h]")
plt.xlabel("時間[h]")
plt.ylim(0, 2000)
plt.grid()

plt.subplot(2, 4, 5)
plt.plot(range(8760), sorted_H_df_output2['E_H_d_t [MJ/h]'], color = 'orange')
plt.ylabel("一次エネルギー消費量[MJ/h]")
plt.xlabel("時間[h]")
plt.ylim(0, 50)
plt.grid()

plt.subplot(2, 4, 6)
plt.plot(range(8760), sorted_H_df_output2['E_UT_H_d_t [MJ/h]'], color = 'orange')
plt.ylabel("未処理負荷（一次エネルギー相当分）[MJ/h]")
plt.xlabel("時間[h]")
plt.ylim(0, 50)
plt.grid()

plt.subplot(2, 4, 7)
plt.plot(range(8760), sorted_H_df_output2['E_E_H_d_t [kWh/h]'], color = 'orange')
plt.ylabel("AC+FAN消費電力[kWh/h]")
plt.xlabel("時間[h]")
plt.ylim(0, 5)
plt.grid()

plt.subplot(2, 4, 8)
plt.plot(range(8760), sorted_H_df_output2['E_E_fan_H_d_t [kWh/h]'], color = 'orange')
plt.ylabel("FAN消費電力[kWh/h]")
plt.xlabel("時間[h]")
plt.ylim(0, 5)
plt.grid()
plt.show()

####Cooling#############################################################################################################

plt.figure(figsize=(25, 7))

plt.subplot(2, 2, 1)
plt.plot(summer_df_output2.index,
         summer_df_output2["Theta_ex_d_t [℃]"],
         label = "外気温[℃]", color = 'lightblue')
plt.plot(summer_df_output5_C.index,
         summer_df_output5_C["X_ex_d_t"] * 1000,
         label = "外気絶対湿度[g/kg']", color = 'orange')
plt.ylabel("温度[℃]、絶対湿度[g/kg']")
plt.ylim(-10, 40)
plt.grid()
plt.legend()

plt.subplot(2, 2, 2)
plt.plot(summer_df_output2.index,
         (summer_df_output2["q_hs_CS_d_t [Wh/h]"] + summer_df_output2["q_hs_CL_d_t [Wh/h]"]) / 1000 ,
         label = "処理熱量[kWh/h]", color = 'red')
plt.plot(summer_df_output2.index,
         summer_df_output2["E_E_C_d_t [kWh/h]"],
         label = "AC+FAN消費電力[kWh/h]", color = 'blue')
plt.plot(summer_df_output2.index,
         summer_df_output2["E_E_fan_C_d_t [kWh/h]"],
         label = "FAN消費電力[kWh/h]", color = 'green')
plt.ylabel("処理熱量[kWh/h]、消費電力[kWh/h]")
plt.ylim(0, 10)
plt.grid()
plt.legend()

plt.subplot(2, 2, 3)
plt.plot(summer_df_output2.index,
         summer_df_output2["E_UT_C_d_t [MJ/h]"],
         label = "処理熱量[kWh/h]", color = 'red')
plt.ylabel("未処理負荷（一次エネルギー相当分）[MJ/h]")
plt.ylim(0, 10)
plt.grid()
plt.legend()

plt.subplot(2, 2, 4)
plt.plot(summer_df_output2.index,
         (summer_df_output2["q_hs_CS_d_t [Wh/h]"] + summer_df_output2["q_hs_CL_d_t [Wh/h]"]) / 1000 / summer_df_output2["E_E_C_d_t [kWh/h]"],
         label = "sCOP[-]", color = 'black')
plt.ylabel("sCOP[-]")
plt.ylim(0, 10)
plt.grid()
plt.legend()
plt.show()

########################################################################################################################

plt.figure(figsize=(25, 7))

plt.subplot(2, 4, 1)
plt.plot(range(8760), sorted_C_df_output2['q_hs_C_d_t [Wh/h]'] / 1000, color = 'lightblue')
plt.ylabel("処理熱量[kWh/h]")
plt.xlabel("時間[h]")
plt.ylim(0, 10)
plt.grid()

plt.subplot(2, 4, 2)
plt.plot(range(8760), sorted_C_df_output2['Theta_hs_C_out_d_t [℃]'], color = 'lightblue')
plt.ylabel("室内機出口温度[℃]")
plt.xlabel("時間[h]")
plt.ylim(0, 50)
plt.grid()

plt.subplot(2, 4, 3)
plt.plot(range(8760), sorted_C_df_output2['Theta_hs_C_in_d_t [℃]'], color = 'lightblue')
plt.ylabel("室内機入口温度[℃]")
plt.xlabel("時間[h]")
plt.ylim(0, 50)
plt.grid()

plt.subplot(2, 4, 4)
plt.plot(range(8760), sorted_C_df_output2['V_hs_supply_C_d_t [m3/h]'], color = 'lightblue')
plt.ylabel("供給風量[m3/h]")
plt.xlabel("時間[h]")
plt.ylim(0, 2000)
plt.grid()

plt.subplot(2, 4, 5)
plt.plot(range(8760), sorted_C_df_output2['E_C_d_t [MJ/h]'], color = 'lightblue')
plt.ylabel("一次エネルギー消費量[MJ/h]")
plt.xlabel("時間[h]")
plt.ylim(0, 50)
plt.grid()

plt.subplot(2, 4, 6)
plt.plot(range(8760), sorted_C_df_output2['E_UT_C_d_t [MJ/h]'], color = 'lightblue')
plt.ylabel("未処理負荷（一次エネルギー相当分）[MJ/h]")
plt.xlabel("時間[h]")
plt.ylim(0, 50)
plt.grid()

plt.subplot(2, 4, 7)
plt.plot(range(8760), sorted_C_df_output2['E_E_C_d_t [kWh/h]'], color = 'lightblue')
plt.ylabel("AC+FAN消費電力[kWh/h]")
plt.xlabel("時間[h]")
plt.ylim(0, 5)
plt.grid()

plt.subplot(2, 4, 8)
plt.plot(range(8760), sorted_C_df_output2['E_E_fan_C_d_t [kWh/h]'], color = 'lightblue')
plt.ylabel("FAN消費電力[kWh/h]")
plt.xlabel("時間[h]")
plt.ylim(0, 5)
plt.grid()
plt.show()

###sCOP#################################################################################################################
plt.figure(figsize=(25, 7))
plt.subplot(1, 2, 1)
plt.scatter(winter_df_output2["q_hs_H_d_t [Wh/h]"] / 1000,
            winter_df_output2["q_hs_H_d_t [Wh/h]"] / 1000 / winter_df_output2["E_E_H_d_t [kWh/h]"],
            label = "sCOP[-]", color = 'orange')
plt.ylabel("sCOP[-]")
plt.ylim(0, 10)
plt.xlabel("処理熱量 [kWh/h]")
plt.xlim(0, 10)
plt.grid()
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter((summer_df_output2["q_hs_CS_d_t [Wh/h]"] + summer_df_output2["q_hs_CL_d_t [Wh/h]"]) / 1000,
            (summer_df_output2["q_hs_CS_d_t [Wh/h]"] + summer_df_output2["q_hs_CL_d_t [Wh/h]"]) / 1000 / summer_df_output2["E_E_C_d_t [kWh/h]"],
            label = "sCOP[-]", color = 'lightblue')
plt.ylabel("sCOP[-]")
plt.ylim(0, 10)
plt.xlabel("処理熱量 [kWh/h]")
plt.xlim(0, 10)
plt.grid()
plt.legend()

plt.show()